In [ ]:
import os, glob
import psycopg2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

# ============================================================
# DB 연결 설정
# ============================================================
DB_MAIN = {
    "host": "",
    "port": "",
    "dbname": "",
    "user": "",
    "password": "",
    "table": ""
}

DB_PUBLIC = {
    "host": "",
    "port": "",
    "dbname": "",
    "user": "",
    "password": "",
    "table": ""
}

SAVE_BASE = r""
MVF_DIR = os.path.join(SAVE_BASE, "mvf_results")
TIEDVD_DIR = os.path.join(SAVE_BASE, "tiedvd_results")
os.makedirs(MVF_DIR, exist_ok=True)
os.makedirs(TIEDVD_DIR, exist_ok=True)

START_DATE = pd.Timestamp("2024-11-11")
END_DATE   = pd.Timestamp("2024-11-15")
V_START, V_END, STEP, TOL = 1008, 792, -2.4, 0.8

# ============================================================
# 방전 구간 탐지
# ============================================================
def find_segments(df, col_current="rack_current",
                  smooth_window=61, I_on=2.0, I_off=0.8,
                  min_samples=30, bridge_idle_samples=10):
    cur = df[col_current].astype(float).to_numpy()
    s = pd.Series(cur).rolling(smooth_window, min_periods=1, center=True).median()
    s = s.rolling(5, min_periods=1, center=True).mean().to_numpy()
    state, start, idle_count = "idle", None, 0
    segs = []
    for i, val in enumerate(s):
        if state == "idle":
            if val >= I_on: state, start = "charge", i
            elif val <= -I_on: state, start = "discharge", i
        elif state == "charge":
            if val > I_off: idle_count = 0
            else:
                idle_count += 1
                if idle_count > bridge_idle_samples:
                    end = i - idle_count
                    if end - start + 1 >= min_samples: segs.append(("charge", start, end))
                    state, start, idle_count = "idle", None, 0
        elif state == "discharge":
            if val < -I_off: idle_count = 0
            else:
                idle_count += 1
                if idle_count > bridge_idle_samples:
                    end = i - idle_count
                    if end - start + 1 >= min_samples: segs.append(("discharge", start, end))
                    state, start, idle_count = "idle", None, 0
    if start is not None:
        end = len(s) - 1
        segs.append((state, start, end))
    return [seg for seg in segs if seg[0] == "discharge"]

# ============================================================
# MVF 계산
# ============================================================
def compute_mvf_from_discharge(df_discharge,
                               voltage_start=1008,
                               voltage_end=792,
                               step=-2.4,
                               seconds=90,
                               tolerance=2):
    if df_discharge.empty:
        return pd.DataFrame(columns=["rack_voltage", "MVF (ΔV avg)"])
    v = df_discharge["rack_voltage"].to_numpy()
    voltage_grid = np.arange(voltage_start, voltage_end + step, step)
    mvf_vals = np.full(len(voltage_grid), np.nan)
    for i, target in enumerate(voltage_grid):
        idx_candidates = np.where(v <= target)[0]
        if len(idx_candidates) == 0: continue
        idx0 = idx_candidates[0]
        if abs(v[idx0] - target) > tolerance: continue
        idx_end = idx0 + seconds
        if idx_end >= len(v): continue
        diffs = v[idx0] - v[idx0 + np.arange(1, seconds + 1)]
        mvf_vals[i] = np.nanmean(diffs)
    mvf_df = pd.DataFrame({"rack_voltage": voltage_grid, "MVF (ΔV avg)": mvf_vals})
    mvf_df.sort_values("rack_voltage", ascending=False, inplace=True)
    return mvf_df

# ============================================================
# TIEDVD 계산
# ============================================================
def compute_tiedvd(df_discharge, v_start, v_end, step, tolerance):
    if df_discharge.empty:
        return pd.DataFrame(columns=["rack_voltage", "TIEDVD_time (s)"])
    df = df_discharge.copy().reset_index(drop=True)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["t_sec"] = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds()
    v = df["rack_voltage"].to_numpy()
    t = df["t_sec"].to_numpy()
    voltage_grid = np.arange(v_start, v_end + step, step)
    results = {"rack_voltage": [], "TIEDVD_time (s)": []}
    for i in range(len(voltage_grid) - 1):
        v_high, v_low = voltage_grid[i], voltage_grid[i + 1]
        idx_high = np.where(np.abs(v - v_high) <= tolerance)[0]
        idx_low  = np.where(np.abs(v - v_low) <= tolerance)[0]
        if len(idx_high) == 0 or len(idx_low) == 0: continue
        t1, t2 = t[idx_high[0]], t[idx_low[0]]
        delta_t = t2 - t1
        if delta_t <= 0: continue
        results["rack_voltage"].append(v_high)
        results["TIEDVD_time (s)"].append(delta_t)
    return pd.DataFrame(results)

# ============================================================
# 평균 전압 상승률 계산
# ============================================================
def compute_mean_voltage_rate(result_df):
    rate_cols = [c for c in result_df.columns if "상승률_" in c]
    mean_values = []
    for c in rate_cols:
        mean_values.append(np.nanmean(result_df[c] * 100))
    summary = pd.DataFrame({
        "DATE": [c.replace("상승률_", "") for c in rate_cols],
        "평균전압상승률(%)": mean_values
    })
    return summary

def align_and_compare_mvf(data_dir, tol=0.5):
    csv_files = sorted(glob.glob(os.path.join(data_dir, "MVF_2024-11-*.csv")))
    if not csv_files: return None
    base_path = [f for f in csv_files if "2024-11-11" in f][0]
    base_df = pd.read_csv(base_path)[["rack_voltage", "MVF (ΔV avg)"]].dropna()
    base_df.rename(columns={"MVF (ΔV avg)": "MVF_2024-11-11"}, inplace=True)
    base_df.sort_values("rack_voltage", ascending=False, inplace=True)
    base_voltages = base_df["rack_voltage"].to_numpy()
    result_df = base_df.copy()

    # 정합형
    for f in csv_files:
        if "2024-11-11" in f: continue
        date = os.path.basename(f).split(".")[0].split("_")[-1]
        df_day = pd.read_csv(f)[["rack_voltage", "MVF (ΔV avg)"]].dropna()
        aligned_mvf = []
        for v_ref in base_voltages:
            near = df_day.loc[np.abs(df_day["rack_voltage"] - v_ref) <= tol, "MVF (ΔV avg)"]
            aligned_mvf.append(near.iloc[0] if len(near) > 0 else np.nan)
        result_df[f"MVF_{date}"] = aligned_mvf
        result_df[f"상승률_{date}"] = (result_df[f"MVF_{date}"] - result_df["MVF_2024-11-11"]) / result_df["MVF_2024-11-11"]

    # 세로형
    merged_df = pd.DataFrame()
    for f in csv_files:
        if "2024-11-11" in f: continue
        date = os.path.basename(f).split(".")[0].split("_")[-1]
        df_day = pd.read_csv(f)[["timestamp", "rack_voltage", "MVF (ΔV avg)"]].dropna()
        aligned_rate, aligned_time, aligned_voltage = [], [], []
        for i, v_ref in enumerate(base_voltages):
            near = df_day.loc[np.abs(df_day["rack_voltage"] - v_ref) <= tol]
            if not near.empty:
                mvf_day = near.iloc[0]["MVF (ΔV avg)"]
                mvf_base = base_df.iloc[i]["MVF_2024-11-11"]
                rate = (mvf_day - mvf_base) / mvf_base if mvf_base != 0 else np.nan
                aligned_rate.append(rate); aligned_time.append(near.iloc[0]["timestamp"]); aligned_voltage.append(v_ref)
            else:
                aligned_rate.append(np.nan); aligned_time.append(np.nan); aligned_voltage.append(v_ref)
        temp_df = pd.DataFrame({"timestamp": aligned_time, "rack_voltage": aligned_voltage, "상승률": aligned_rate, "DATE": date})
        merged_df = pd.concat([merged_df, temp_df], ignore_index=True)

    # 저장
    result_df.to_csv(os.path.join(data_dir, "MVF_상승률_정합.csv"), index=False, encoding="utf-8-sig")
    merged_df.to_csv(os.path.join(data_dir, "MVF_상승률_세로형.csv"), index=False, encoding="utf-8-sig")
    print(f"[저장 완료] MVF 상승률 2종 (정합형+세로형)")
    summary = compute_mean_voltage_rate(result_df)
    print(summary)
    return result_df, summary

# ============================================================
# 방전시간단축률 계산
# ============================================================
def align_and_compare_tiedvd(data_dir, tol=0.5):
    csv_files = sorted(glob.glob(os.path.join(data_dir, "TIEDVD_2024-11-*.csv")))
    if not csv_files: return None
    base_path = [f for f in csv_files if "2024-11-11" in f][0]
    base_df = pd.read_csv(base_path)[["rack_voltage", "TIEDVD_time (s)"]].dropna()
    base_df.rename(columns={"TIEDVD_time (s)": "TIEDVD_2024-11-11"}, inplace=True)
    base_df.sort_values("rack_voltage", ascending=False, inplace=True)
    base_voltages = base_df["rack_voltage"].to_numpy()
    result_df = base_df.copy()
    for f in csv_files:
        if "2024-11-11" in f: continue
        date = os.path.basename(f).split(".")[0].split("_")[-1]
        df_day = pd.read_csv(f)[["rack_voltage", "TIEDVD_time (s)"]].dropna()
        aligned_tiedvd = []
        for v_ref in base_voltages:
            near = df_day.loc[np.abs(df_day["rack_voltage"] - v_ref) <= tol, "TIEDVD_time (s)"]
            aligned_tiedvd.append(near.iloc[0] if len(near) > 0 else np.nan)
        result_df[f"TIEDVD_{date}"] = aligned_tiedvd

    dates = sorted([os.path.basename(f).split(".")[0].split("_")[-1] for f in csv_files])
    for i in range(len(dates) - 1):
        d1, d2 = dates[i], dates[i + 1]
        col1, col2 = f"TIEDVD_{d1}", f"TIEDVD_{d2}"
        if col1 in result_df and col2 in result_df:
            result_df[f"상승률_{d2}"] = (result_df[col2] - result_df[col1]) / result_df[col1] * 100
    result_df.to_csv(os.path.join(data_dir, "TIEDVD_상승률_정합.csv"), index=False, encoding="utf-8-sig")
    print(f"[저장 완료] TIEDVD 상승률 정합 파일")
    return result_df

# ============================================================
# 메인 실행
# ============================================================
if __name__ == "__main__":
    # (1) 평균전압상승률
    mvf_df, mvf_summary = align_and_compare_mvf(MVF_DIR)
    # (2) 방전시간단축률
    tiedvd_df = align_and_compare_tiedvd(TIEDVD_DIR)
    # (3) HI 지표 조회
    hi_df = fetch_hi_from_public_db()

In [ ]:
import os, glob
import psycopg2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta
from scipy.special import expit  # Sigmoid

# ============================================================
# DB 연결 설정
# ============================================================
DB_MAIN = {
    "host": "",
    "port": "",
    "dbname": "",
    "user": "",
    "password": "",
    "table": ""
}

DB_PUBLIC = {
    "host": "",
    "port": "",
    "dbname": "",
    "user": "",
    "password": "",
    "table": ""
}

SAVE_BASE = r""
os.makedirs(SAVE_BASE, exist_ok=True)

# ============================================================
# Adaptive Sigmoid 함수 정의
# ============================================================
def adaptive_sigmoid(x, mean, std, k=5):
    """
    평균과 표준편차 기반으로 적응형 Sigmoid 정규화 수행
    → 값이 평균보다 크면 급격히 1로, 작으면 0으로 수렴
    """
    if np.isnan(x): return np.nan
    return expit(k * ((x - mean) / (std + 1e-12)))

# ============================================================
# SOS 계산 함수
# ============================================================
def compute_sos_from_analysis_table():
    print("[INFO] SOS 계산 시작")

    conn = psycopg2.connect(
        host=DB_PUBLIC["host"],
        port=DB_PUBLIC["port"],
        dbname=DB_PUBLIC["dbname"],
        user=DB_PUBLIC["user"],
        password=DB_PUBLIC["password"]
    )

    query = """
    SET TIME ZONE 'Asia/Seoul';
    SELECT "TAG", "TIMESTAMP", "VALUE"
    FROM analysis_result_cnu
    WHERE "TAG" IN ('TIECVD_DR', 'VIECTD_IR', 'MVF_IR')
    ORDER BY "TIMESTAMP" ASC;
    """

    df = pd.read_sql_query(query, conn)
    df["TIMESTAMP"] = pd.to_datetime(df["TIMESTAMP"], utc=True).dt.tz_convert("Asia/Seoul")

    # 태그별 그룹화
    grouped = {tag: df[df["TAG"] == tag].copy() for tag in df["TAG"].unique()}

    # 각 태그별 정규화 수행
    for tag, sub in grouped.items():
        m, s = sub["VALUE"].mean(), sub["VALUE"].std()
        grouped[tag]["NORM"] = sub["VALUE"].apply(lambda v: adaptive_sigmoid(v, m, s, k=5))

    # 타임스탬프 기준 병합
    df_merged = pd.concat([g[["TIMESTAMP", "TAG", "NORM"]] for g in grouped.values()])
    df_pivot = df_merged.pivot(index="TIMESTAMP", columns="TAG", values="NORM")

    # SOS 계산 (3개 지표 평균)
    df_pivot["SOS"] = df_pivot.mean(axis=1)
    df_pivot.reset_index(inplace=True)

    # sos 테이블 저장 (없으면 생성)
    with conn.cursor() as cur:
        cur.execute("""
        CREATE TABLE IF NOT EXISTS sos (
            timestamp TIMESTAMPTZ PRIMARY KEY,
            sos_value DOUBLE PRECISION
        );
        """)
        conn.commit()

        for _, row in df_pivot.iterrows():
            cur.execute("""
                INSERT INTO sos (timestamp, sos_value)
                VALUES (%s, %s)
                ON CONFLICT (timestamp) DO UPDATE
                SET sos_value = EXCLUDED.sos_value;
            """, (row["TIMESTAMP"], row["SOS"]))
        conn.commit()

    conn.close()
    print(f"[완료] SOS 계산 및 저장 완료 ({len(df_pivot)}행)")
    return df_pivot

# ============================================================
# 시각화
# ============================================================
def plot_sos(df_sos):
    plt.figure(figsize=(12,4))
    plt.plot(df_sos["TIMESTAMP"], df_sos["SOS"], color='tab:red', linewidth=1.8, label="SOS")
    plt.title("State of Safety (SOS) Score", fontsize=13)
    plt.xlabel("Timestamp (Asia/Seoul)")
    plt.ylabel("Safety Score")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()

# ============================================================
# 실행
# ============================================================
if __name__ == "__main__":
    df_sos = compute_sos_from_analysis_table()
    plot_sos(df_sos)